# Exercise 2. LoRa for Low-Resource Languages
NLP for social good is not just about reducing harmful outputs; it is also about making AI accessible across languages, not only English. Low- and medium-resource languages, from Nigerian Pidgin to Danish, are often left behind. 

```{figure} ../figures/class8/neural-space-low-resource.png
---
name: neural-space-low-resource
width: 100%
---
Fig. borrowed from [NeuralSpace blogpost](https://medium.com/neuralspace/challenges-in-using-nlp-for-low-resource-languages-and-how-neuralspace-solves-them-54a01356a71b) by Felix Laumann
```

Fine-tuning LLMs can help, but it is costly. LoRA (Low-Rank Adaptation) offers a parameter-efficient alternative, reducing trainable parameters by up to 10,000 times. In other words, rather than training all 8 billion parameters of a model like [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B-Base), LoRA updates only a small fraction.

## 2.1 Intro to LoRa?
When we're doing LoRa, we are esentially training a LoRa adapter that could technically be placed on other models (if the base architecture matches):

```{figure} ../figures/class8/lora_adapter.png
---
name: lora_adapter
width: 100%
---
From HF's [smol course](https://huggingface.co/learn/smol-course/en/unit1/3a)
```

If you're interested in the math behind this (but in an intuitive way), I encourage you to read Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/i/138081202/a-brief-introduction-to-lora). You can also read the original paper by {cite:t}`hu_lora_2021`. 

## 2.2 Setup
For the code implementation, we'll use the [PEFT](https://huggingface.co/docs/peft/en/index) and [TRL](https://huggingface.co/docs/trl/en/index) library by Hugging Face
```bash
source .venv/bin/activate
pip install peft trl
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers datasets
```

Let's import:

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

## 2.3 Load Model & Data
For today's exercise, we'll try to make a smaller version of `SmolLM2` good at English to Danish machine translation

:::{admonition} You can use LoRa for much more than Translation :)
:class: dropdown, tip
As a simple introduction to LoRA, we're doing machine translation, but you can use this approach for anything you'd like really - feel free to switch out the dataset for something you'd like. Or use this notebook as a inspiration for the exam :).

See also this tutorial for instruction-tuning a danish language model using QLoRA -> [Tutorial: Finetuning Language Models](https://www.foundationmodels.dk/blog/2024/02/02/tutorial-finetuning-language-models.html)
:::

We'll load a smaller version of `SmolLM2`:

In [3]:
model_id = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

We'll load the Danish-English translation set:

In [4]:
train_ds = load_dataset("Helsinki-NLP/opus-100", "da-en", split="train")

And downsample:

In [5]:
train_ds = train_ds.train_test_split(seed=24, test_size=5000)["test"]

Let's look at the only column, "translation" to see how it is structured: 

In [6]:
train_ds["translation"]

Column([{'da': 'Men den plan var ikke idiotsikker.', 'en': "Well, that plan wasn't foolproof."}, {'da': 'Det mener jeg, at Europa-Parlamentet bør notere sig, og det bør støtte ønsket om også på denne måde at deltage fuldt ud i det internationale samfund, for hvem ved, om domstolen i fremtiden netop kommer til at dømme de forbrydelser, der er blevet begået i landet.', 'en': 'Well, I feel that Parliament should take note and encourage this desire to become a fully paid-up member of the international community, also in this respect, because who knows, in the future the Court could operate precisely to try crimes committed in that country.'}, {'da': 'virkning på Deres blodsukkerkontrol. i', 'en': 'effect on your blood glucose control. ed'}, {'da': 'Du er så smuk.', 'en': "You're so beautiful."}, {'da': 'Jeg kom netop med to eksempler - en i Messinastrædet og en på Seinen for et par dage siden.', 'en': 'I gave just two examples: one on the Strait of Messina and the other on the River Seine 

Let's print a few:

In [7]:
for translation in train_ds["translation"][:5]:
    print(f"EN: {translation['en']}")
    print(f"DA: {translation['da']}")
    print()

EN: Well, that plan wasn't foolproof.
DA: Men den plan var ikke idiotsikker.

EN: Well, I feel that Parliament should take note and encourage this desire to become a fully paid-up member of the international community, also in this respect, because who knows, in the future the Court could operate precisely to try crimes committed in that country.
DA: Det mener jeg, at Europa-Parlamentet bør notere sig, og det bør støtte ønsket om også på denne måde at deltage fuldt ud i det internationale samfund, for hvem ved, om domstolen i fremtiden netop kommer til at dømme de forbrydelser, der er blevet begået i landet.

EN: effect on your blood glucose control. ed
DA: virkning på Deres blodsukkerkontrol. i

EN: You're so beautiful.
DA: Du er så smuk.

EN: I gave just two examples: one on the Strait of Messina and the other on the River Seine a few days ago.
DA: Jeg kom netop med to eksempler - en i Messinastrædet og en på Seinen for et par dage siden.



## 1.3 Prompt Templating
We want a `prompt` column that inserts the English sentence as the `Source` and a `completion` column that has the Danish example in this formatting by{cite:t}`alves_steering_2023`:
```{figure} ../figures/class8/prompt-template-alves.png
---
name: prompt-template-alves
width: 80%
---
Prompt template by {cite:t}`alves_steering_2023`
```
X should be "English" and Y should be "Danish" in our context.

### Your Turn: Formatting the Prompt
:::{admonition} HANDS-ON
:class: red
1. Create a function called `def format_prompt(example)`
    - It should process a single row `example` in our dataset
    - Use the prompt template above to create a `prompt` with English as the source and a `completion` column with the target Danish sentence.
    - Return a dictionary entry `{"prompt": prompt, "completion": completion}`

2. Test the function on a single example in `train_ds`, printing the prompt and completion"
:::

#### Solution

In [9]:
# define function
def format_prompt(example):
    translation = example["translation"]
    prompt = f"Translate the source text from English to Danish. Source: {translation['en']} Target: "
    completion = f"{translation['da']}"

    prompt_completion = {"prompt": prompt, "completion": completion}
    return prompt_completion

# test on one example
example = train_ds[0]
print(format_prompt(example)["prompt"])
print(format_prompt(example)["completion"])

Translate the source text from English to Danish. Source: Well, that plan wasn't foolproof. Target: 
Men den plan var ikke idiotsikker.


### Adding a Prompt Column 
We can now add the prompt column to our ds using our new `format_prompt` column:

In [10]:
train_ds = train_ds.map(format_prompt, batched=False)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [11]:
print(train_ds["prompt"][0])
print(train_ds["completion"][0])

Translate the source text from English to Danish. Source: Well, that plan wasn't foolproof. Target: 
Men den plan var ikke idiotsikker.


Let's tokenize this, modifying our tokenize function from [class 5](/book/class5/001_finetune.ipynb):

In [12]:
def preprocess_function(examples):
    """Tokenize the prompts without truncation"""
    return tokenizer(examples["prompt"], text_target=examples["completion"]) 

tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=["translation"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

## 1.4 LoRa Config & Training
We'll start by configuring LoRA: 

In [13]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [15]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(output_dir="lora-adapter", num_train_epochs=1, per_device_train_batch_size=4, packing=True),
    train_dataset=tokenized_train,
    peft_config=peft_config,
)
trainer.train()

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn

Step,Training Loss
10,4.128000
20,4.121300
30,4.195600


TrainOutput(global_step=37, training_loss=4.1394466709446265, metrics={'train_runtime': 151.4043, 'train_samples_per_second': 0.958, 'train_steps_per_second': 0.244, 'total_flos': 93009737932416.0, 'train_loss': 4.1394466709446265, 'entropy': 4.315137454441616, 'num_tokens': 145331.0, 'mean_token_accuracy': 0.28696590662002563, 'epoch': 1.0})

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForCausalLM.from_pretrained(base_model_name)

# load the trained LoRA adapters
model = PeftModel.from_pretrained(model, "lora-adapter/checkpoint-51")

# inference
prompt = "Translate the source text from English to Danish. Source: I love coffee. Target: "
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Translate the source text from English to Danish. Source: I love coffee. Target: 1000 words.

The first thing I do when I get a new word is to look it up in the dictionary. I don’t know what it means, but I know it’s a word that I’m going
